In [1]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
print("Environment configured - protobuf set to pure Python mode, TensorFlow disabled for transformers")

Environment configured - protobuf set to pure Python mode, TensorFlow disabled for transformers


In [2]:
# Installation (run once)
!pip install -q PyMuPDF pytesseract pillow
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q langchain-google-genai sentence-transformers faiss-cpu

In [3]:
!pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Core imports
import os
import re
import io
import unicodedata
import hashlib
import json
from pathlib import Path
from typing import List, Dict, Optional, Tuple

# Numerical & Image Processing
import langchain_community
import numpy as np
import cv2

# PDF & Image Processing
import fitz  # PyMuPDF
from PIL import Image
import pytesseract

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.chains import create_retrieval_chain

from langchain.prompts import PromptTemplate

print("All libraries imported successfully")

C:\Users\quoch\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully


In [3]:
# Configuration Class
class RAGConfig:
    """Centralized configuration for RAG system"""
    
    # Paths
    BASE_DIR = Path.cwd()
    PDF_DIR = BASE_DIR / "File_PDFs"
    VECTOR_STORE_DIR = BASE_DIR / "vector_store"
    OCR_DIR = BASE_DIR / "OCR"
    OCR_CACHE_DIR = VECTOR_STORE_DIR / "ocr_cache"
    
    # OCR Settings
    TESSERACT_CMD = str(OCR_DIR / "tesseract.exe")
    TESSDATA_PREFIX = str(OCR_DIR / "tessdata")
    OCR_LANGUAGES = "vie+eng"
    OCR_DPI = 2  # Matrix scale factor
    OCR_PSM = 3  # Page segmentation mode (3=auto, 6=uniform text block)
    OCR_OEM = 1  # OCR Engine Mode (1=LSTM neural net, 3=default+LSTM)
    
    # OCR Preprocessing
    OCR_PREPROCESSING_ENABLED = True
    OCR_ADAPTIVE_THRESHOLD = True
    OCR_DENOISE = True
    OCR_DESKEW = True
    
    # OCR Post-processing
    OCR_VIETNAMESE_CORRECTION = True
    OCR_CORRECTION_AGGRESSIVE = False  # Conservative by default
    
    # OCR Quality & Caching
    OCR_CONFIDENCE_THRESHOLD = 0.60  # Retry threshold for low confidence
    OCR_CACHE_ENABLED = True
    OCR_MAX_RETRY_ATTEMPTS = 2
    
    # Text Processing
    CHUNK_SIZE = 1000
    CHUNK_OVERLAP = 200
    SEPARATORS = ["\n\n", "\n", ". ", " ", ""]
    
    # Embeddings
    EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
    EMBEDDING_DEVICE = "cpu"  # Change to "cuda" if GPU available
    
    # LLM
    LLM_MODEL = "gemini-2.5-flash"
    LLM_TEMPERATURE = 0.2
    LLM_MAX_TOKENS = 1024
    
    # Retrieval
    RETRIEVAL_K = 6  # Increase recall to avoid missing answer-bearing chunks
    
    # API Keys (Set via environment variables)
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "YOUR_API_KEY_HERE")

config = RAGConfig()
print(" Configuration loaded")
print(f" Base directory: {config.BASE_DIR}")
print(f" PDF directory: {config.PDF_DIR}")

 Configuration loaded
 Base directory: a:\RAG-SGU
 PDF directory: a:\RAG-SGU\File_PDFs


## 2️⃣ Data Ingestion - OCR & Text Processing

In [4]:
class OCRProcessor:
    """Handle OCR processing with Tesseract"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self._setup_tesseract()
    
    def _setup_tesseract(self):
        """Configure Tesseract paths"""
        if os.path.exists(self.config.TESSERACT_CMD):
            pytesseract.pytesseract.tesseract_cmd = self.config.TESSERACT_CMD
            os.environ['TESSDATA_PREFIX'] = self.config.TESSDATA_PREFIX + os.sep
            print("✅ Tesseract configured successfully")
        else:
            raise FileNotFoundError(f"Tesseract not found at: {self.config.TESSERACT_CMD}")
    
    def _pixmap_to_cv2(self, pixmap) -> np.ndarray:
        """Convert PyMuPDF pixmap to OpenCV array
        
        Args:
            pixmap: fitz.Pixmap object
            
        Returns:
            numpy array in BGR format suitable for OpenCV
        """
        img_data = pixmap.tobytes("png")
        nparr = np.frombuffer(img_data, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        return img
    
    def _detect_skew(self, gray: np.ndarray) -> float:
        """Detect rotation angle of image
        
        Args:
            gray: Grayscale image array
            
        Returns:
            Detected skew angle in degrees
        """
        # Use Hough Line Transform to detect dominant lines
        edges = cv2.Canny(gray, 50, 150, apertureSize=3)
        lines = cv2.HoughLines(edges, 1, np.pi/180, 200)
        
        if lines is None:
            return 0.0
        
        # Calculate angles
        angles = []
        for rho, theta in lines[:, 0]:
            angle = np.degrees(theta) - 90
            if -45 < angle < 45:  # Filter out vertical/horizontal lines
                angles.append(angle)
        
        if not angles:
            return 0.0
        
        # Return median angle
        return float(np.median(angles))
    
    def _rotate_image(self, image: np.ndarray, angle: float) -> np.ndarray:
        """Rotate image to correct skew
        
        Args:
            image: Input image array
            angle: Rotation angle in degrees
            
        Returns:
            Rotated image array
        """
        if abs(angle) < 0.5:  # Skip rotation for very small angles
            return image
        
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        
        # Get rotation matrix
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        
        # Perform rotation
        rotated = cv2.warpAffine(image, M, (w, h),
                                 flags=cv2.INTER_CUBIC,
                                 borderMode=cv2.BORDER_REPLICATE)
        return rotated
    
    def _preprocess_image(self, image: Image.Image) -> np.ndarray:
        """Apply preprocessing pipeline to improve OCR quality
        
        Args:
            image: PIL Image object
            
        Returns:
            Preprocessed grayscale numpy array
        """
        # Convert PIL Image to OpenCV array
        img_array = np.array(image)
        
        # Convert to grayscale
        if len(img_array.shape) == 3:
            gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        else:
            gray = img_array
        
        # Apply adaptive thresholding
        if self.config.OCR_ADAPTIVE_THRESHOLD:
            gray = cv2.adaptiveThreshold(
                gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY, 11, 2
            )
        
        # Apply denoising
        if self.config.OCR_DENOISE:
            gray = cv2.fastNlMeansDenoising(gray, h=10)
        
        # Detect and correct skew
        if self.config.OCR_DESKEW:
            angle = self._detect_skew(gray)
            if abs(angle) > 0.5:
                gray = self._rotate_image(gray, angle)
        
        return gray
    
    @staticmethod
    def clean_text(text: str) -> str:
        """Clean and normalize text"""
        if not text:
            return ""
        text = unicodedata.normalize('NFC', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def process_pdf(self, pdf_path: str, verbose: bool = True) -> str:
        """Extract text from a single PDF using OCR"""
        pdf_path = Path(pdf_path)
        
        if not pdf_path.exists():
            raise FileNotFoundError(f"PDF not found: {pdf_path}")
        
        if verbose:
            print(f"\n📄 Processing: {pdf_path.name}")
        
        full_text = ""
        
        try:
            doc = fitz.open(str(pdf_path))
            total_pages = len(doc)
            
            if verbose:
                print(f"   📖 Total pages: {total_pages}")
            
            for page_num, page in enumerate(doc, 1):
                # Convert page to high-res image
                pix = page.get_pixmap(matrix=fitz.Matrix(self.config.OCR_DPI, self.config.OCR_DPI))
                img_data = pix.tobytes("png")
                img = Image.open(io.BytesIO(img_data))
                
                # OCR
                text = pytesseract.image_to_string(img, lang=self.config.OCR_LANGUAGES)
                full_text += text + "\n"
                
                # Progress
                if verbose and page_num % 10 == 0:
                    print(f"      ⏳ Processed: {page_num}/{total_pages} pages")
            
            doc.close()
            cleaned_text = self.clean_text(full_text)
            
            if verbose:
                print(f"   ✅ Extraction complete: {len(cleaned_text):,} characters")
            
            return cleaned_text
            
        except Exception as e:
            raise RuntimeError(f"OCR processing failed for {pdf_path.name}: {e}")
    
    def process_directory(self, directory: str, pattern: str = "*.pdf", verbose: bool = True) -> Dict[str, str]:
        """Extract text from all PDFs in a directory
        
        Args:
            directory: Path to directory containing PDFs
            pattern: Glob pattern for PDF files (default: "*.pdf")
            verbose: Print progress information
            
        Returns:
            Dictionary mapping filename to extracted text
        """
        directory = Path(directory)
        
        if not directory.exists():
            raise FileNotFoundError(f"Directory not found: {directory}")
        
        # Find all PDF files
        pdf_files = sorted(directory.glob(pattern))
        
        if not pdf_files:
            print(f"⚠️ No PDF files found in {directory}")
            return {}
        
        print(f"\n{'='*70}")
        print(f"📁 Processing Directory: {directory.name}")
        print(f"{'='*70}")
        print(f"   Found {len(pdf_files)} PDF file(s)")
        print(f"{'='*70}\n")
        
        results = {}
        
        for idx, pdf_file in enumerate(pdf_files, 1):
            print(f"[{idx}/{len(pdf_files)}] Processing: {pdf_file.name}")
            
            try:
                text = self.process_pdf(pdf_file, verbose=verbose)
                results[pdf_file.name] = text
                print(f"   ✅ Success: {len(text):,} characters extracted\n")
                
            except Exception as e:
                print(f"   ❌ Error: {e}\n")
                results[pdf_file.name] = ""
        
        # Summary
        successful = sum(1 for text in results.values() if text)
        total_chars = sum(len(text) for text in results.values())
        
        print(f"\n{'='*70}")
        print(f"📊 PROCESSING SUMMARY")
        print(f"{'='*70}")
        print(f"   Total files: {len(pdf_files)}")
        print(f"   Successful: {successful}")
        print(f"   Failed: {len(pdf_files) - successful}")
        print(f"   Total characters: {total_chars:,}")
        print(f"{'='*70}\n")
        
        return results
    
    def merge_texts(self, texts_dict: Dict[str, str], add_separators: bool = True) -> str:
        """Merge texts from multiple PDFs
        
        Args:
            texts_dict: Dictionary mapping filename to text
            add_separators: Add document separators between texts
            
        Returns:
            Combined text from all documents
        """
        if not texts_dict:
            return ""
        
        combined_text = ""
        
        for filename, text in texts_dict.items():
            if not text:
                continue
            
            if add_separators:
                # Add separator with filename
                separator = f"\n\n{'='*50}\n"
                separator += f"DOCUMENT: {filename}\n"
                separator += f"{'='*50}\n\n"
                combined_text += separator
            
            combined_text += text + "\n\n"
        
        return self.clean_text(combined_text)

# Initialize OCR processor
ocr_processor = OCRProcessor(config)
print("✅ OCR Processor initialized")

✅ Tesseract configured successfully
✅ OCR Processor initialized


In [5]:
# Process ALL PDF documents in directory
print(f"📂 Target directory: {config.PDF_DIR}")

if config.PDF_DIR.exists():
    # Process all PDFs in directory instead of a single hard-coded file
    texts_dict = ocr_processor.process_directory(config.PDF_DIR, pattern="*.pdf", verbose=True)

    # Keep only successfully extracted documents
    normalized_texts_dict: Dict[str, str] = {
        str(filename): str(content)
        for filename, content in texts_dict.items()
        if isinstance(content, str) and content.strip()
    }

    if not normalized_texts_dict:
        raise RuntimeError("No extractable text found from PDF directory")

    texts_dict = normalized_texts_dict
    raw_text = ocr_processor.merge_texts(texts_dict, add_separators=True)
    
    # Statistics
    print(f"📊 FINAL STATISTICS:")
    print(f"{'='*70}")
    print(f"   Total documents: {len(texts_dict)}")
    print(f"   Total characters: {len(raw_text):,}")
    print(f"   Total words (approx): {len(raw_text.split()):,}")
    
    # Document breakdown
    print(f"\n   Documents processed:")
    for filename, text in texts_dict.items():
        status = "✅" if text else "❌"
        char_count = len(text) if text else 0
        print(f"      {status} {filename}: {char_count:,} chars")
    
    print(f"\n{'='*70}")
    print(f"\n📝 Preview of combined text:")
    print(raw_text[:500] + "...\n")
    
else:
    print(f"❌ PDF directory not found: {config.PDF_DIR}")
    print(f"   Please create directory and add PDF files")
    raw_text = ""

# Alternative: Process single file (uncomment if needed)
# pdf_file = config.PDF_DIR / "BanMOTa_CNTT_2020-2024.pdf"
# if pdf_file.exists():
#     raw_text = ocr_processor.process_pdf(pdf_file)
# else:
#     print(f"❌ PDF not found: {pdf_file}")

📂 Target directory: a:\RAG-SGU\File_PDFs

📁 Processing Directory: File_PDFs
   Found 2 PDF file(s)

[1/2] Processing: Bản sao của Bản sao của BanMOTa_CNTT_2020-2024.pdf

📄 Processing: Bản sao của Bản sao của BanMOTa_CNTT_2020-2024.pdf
   📖 Total pages: 62
      ⏳ Processed: 10/62 pages
      ⏳ Processed: 20/62 pages
      ⏳ Processed: 30/62 pages
      ⏳ Processed: 40/62 pages
      ⏳ Processed: 50/62 pages
      ⏳ Processed: 60/62 pages
   ✅ Extraction complete: 115,357 characters
   ✅ Success: 115,357 characters extracted

[2/2] Processing: Bản sao của Bản sao của CAMNANGTSVSGU2022.pdf

📄 Processing: Bản sao của Bản sao của CAMNANGTSVSGU2022.pdf
   📖 Total pages: 31
      ⏳ Processed: 10/31 pages
      ⏳ Processed: 20/31 pages
      ⏳ Processed: 30/31 pages
   ✅ Extraction complete: 56,025 characters
   ✅ Success: 56,025 characters extracted


📊 PROCESSING SUMMARY
   Total files: 2
   Successful: 2
   Failed: 0
   Total characters: 171,382

📊 FINAL STATISTICS:
   Total documents: 2
 

### 📁 Hỗ trợ xử lý nhiều PDF files

**Cách sử dụng:**
```python
# Xử lý cả thư mục
texts_dict = ocr_processor.process_directory(config.PDF_DIR)

# Hoặc xử lý 1 file cụ thể
text = ocr_processor.process_pdf("path/to/file.pdf")
```

In [8]:
# 🔧 ADVANCED: Custom processing options (Optional)

# Option 2: Process specific files only
# specific_files = ["file1.pdf", "file2.pdf"]
# texts_dict = {}
# for filename in specific_files:
#     pdf_path = config.PDF_DIR / filename
#     if pdf_path.exists():
#         texts_dict[filename] = ocr_processor.process_pdf(pdf_path)

# Option 3: Filter by filename pattern
# texts_dict = ocr_processor.process_directory(config.PDF_DIR, pattern="*CNTT*.pdf")

# Option 4: Process without document separators (for single-document feel)
# raw_text = ocr_processor.merge_texts(texts_dict, add_separators=False)

print("💡 Tip: Uncomment above code blocks to use custom processing options")

💡 Tip: Uncomment above code blocks to use custom processing options


## 3️⃣ Text Chunking & Processing

In [6]:
class TextChunker:
    """Handle text splitting into chunks"""
    
    def __init__(self, config: RAGConfig):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.CHUNK_SIZE,
            chunk_overlap=config.CHUNK_OVERLAP,
            separators=config.SEPARATORS,
            length_function=len
        )
    
    def split_text(self, text: str, verbose: bool = True) -> List[str]:
        """Split text into chunks"""
        if not text or len(text) < 100:
            raise ValueError("Input text is too short or empty")
        
        chunks = self.splitter.split_text(text)
        
        if verbose:
            print(f"\n Text Chunking:")
            print(f"   Total chunks: {len(chunks)}")
            print(f"   Avg chunk size: {sum(len(c) for c in chunks) // len(chunks)} chars")
            print(f"   Min/Max: {min(len(c) for c in chunks)} / {max(len(c) for c in chunks)} chars")
        
        return chunks

# Create chunks
chunker = TextChunker(config)
chunks = chunker.split_text(raw_text)

print(f"\n Sample chunk:")
print(f"{chunks[0][:200]}...")


 Text Chunking:
   Total chunks: 227
   Avg chunk size: 852 chars
   Min/Max: 114 / 1000 chars

 Sample chunk:
================================================== DOCUMENT: Bản sao của Bản sao của BanMOTa_CNTT_2020-2024.pdf ================================================== UY BAN NHÂN DAN THANH PHO HO CHi MINH...


## 4️⃣ Embedding Model & Vector Store

In [7]:
class VectorStoreManager:
    """Manage embeddings and vector store"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self.embeddings = None
        self.vector_store = None
        self._initialize_embeddings()
    
    def _initialize_embeddings(self):
        """Initialize embedding model"""
        print(f"\n Loading embedding model: {self.config.EMBEDDING_MODEL}")
        
        # Keep transformers on the PyTorch path to avoid TensorFlow ABI issues on Windows.
        os.environ.setdefault('USE_TF', '0')
        os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
        
        try:
            self.embeddings = HuggingFaceEmbeddings(
                model_name=self.config.EMBEDDING_MODEL,
                model_kwargs={'device': self.config.EMBEDDING_DEVICE},
                encode_kwargs={'normalize_embeddings': True}
            )
        except TypeError as exc:
            if "Unable to convert function return value to a Python type" in str(exc):
                raise RuntimeError(
                    "Embedding model init failed because TensorFlow was loaded. "
                    "Restart kernel and run from the first setup cell."
                ) from exc
            raise
        
        print(" Embedding model loaded")
    
    def create_vector_store(self, texts: List[str], verbose: bool = True) -> FAISS:
        """Create FAISS vector store from texts"""
        if verbose:
            print(f"\n Building vector store from {len(texts)} chunks...")
        
        self.vector_store = FAISS.from_texts(
            texts=texts,
            embedding=self.embeddings
        )
        
        if verbose:
            print(" Vector store created successfully")
        
        return self.vector_store
    
    def save_vector_store(self, path: Optional[str] = None):
        """Persist vector store to disk"""
        if self.vector_store is None:
            raise ValueError("No vector store to save")
        
        save_path = path or str(self.config.VECTOR_STORE_DIR)
        os.makedirs(save_path, exist_ok=True)
        
        self.vector_store.save_local(save_path)
        print(f" Vector store saved to: {save_path}")
    
    def load_vector_store(self, path: Optional[str] = None) -> FAISS:
        """Load vector store from disk"""
        load_path = path or str(self.config.VECTOR_STORE_DIR)
        
        if not os.path.exists(load_path):
            raise FileNotFoundError(f"Vector store not found at: {load_path}")
        
        self.vector_store = FAISS.load_local(
            load_path,
            self.embeddings,
            allow_dangerous_deserialization=True
        )
        print(f" Vector store loaded from: {load_path}")
        return self.vector_store

# Initialize vector store manager
vector_manager = VectorStoreManager(config)


 Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


C:\Users\quoch\AppData\Local\Temp\ipykernel_9384\194417861.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


 Embedding model loaded


In [11]:
# Smoke test for embedding + FAISS lifecycle
sample_chunks = [
    "Ngành Công nghệ thông tin có mục tiêu đào tạo kỹ sư phần mềm.",
    "Chương trình bao gồm học phần lập trình, cơ sở dữ liệu và AI.",
    "Sinh viên có thể làm việc ở vị trí backend, frontend, data engineer.",
]

smoke_store = vector_manager.create_vector_store(sample_chunks, verbose=False)
docs = smoke_store.similarity_search("mục tiêu đào tạo ngành CNTT", k=2)
assert len(docs) == 2, "Similarity search returned unexpected number of docs"

smoke_path = config.VECTOR_STORE_DIR / "smoke_test"
vector_manager.save_vector_store(str(smoke_path))
_ = vector_manager.load_vector_store(str(smoke_path))

import shutil
shutil.rmtree(smoke_path, ignore_errors=True)
print("Embedding/vector store smoke test passed")

 Vector store saved to: a:\RAG-SGU\vector_store\smoke_test
 Vector store loaded from: a:\RAG-SGU\vector_store\smoke_test
Embedding/vector store smoke test passed


In [8]:
# Create and save vector store
vector_store = vector_manager.create_vector_store(chunks)

# Save for future use
try:
    vector_manager.save_vector_store()
except Exception as e:
    print(f" Could not save vector store: {e}")


 Building vector store from 227 chunks...
 Vector store created successfully
 Vector store saved to: a:\RAG-SGU\vector_store


In [13]:
# Test similarity search
test_query = "Mục tiêu đào tạo của ngành Công nghệ thông tin là gì?"
print(f"\n Testing retrieval with query: '{test_query}'\n")

relevant_docs = vector_store.similarity_search(test_query, k=3)

for i, doc in enumerate(relevant_docs, 1):
    print(f"\n{'='*60}")
    print(f" Document {i}:")
    print(f"{'='*60}")
    print(doc.page_content[:300] + "...")


 Testing retrieval with query: 'Mục tiêu đào tạo của ngành Công nghệ thông tin là gì?'


 Document 1:
. Cơ hội học tập, nâng cao trình độ sau tốt nghiệp Có khả năng tự nghiên cứu và cập nhật công nghệ mới về lĩnh vực công nghệ thông tin dé nang cao trình độ va đáp tng yêu cau công việc thực tiễn. Có đủ kiến thức dé tiếp tục hoc tiếp lên trình dé thạc sỹ hoặc tiền sỹ ở các ngành: Công nghệ thông tin,...

 Document 2:
. 4.2. Sir mang Dao tao nguén nhân luc ngành Công nghé thông tin phục vu muc tiéu phat trién bén vững. Thực hiện nghién cứu khoa hoc va ứng dung cong nghé dap ứng sy nghiệp công nghiệp hóa, hiện dai hóa dat nước va hội nhập quốc tế. 5. Mục tiêu của CTĐT (POs) ngành Công nghệ thông tin 5.1. Mục tiêu ...

 Document 3:
. KIỀN THỨC A1, Kiến thức chung IO I- Áp dung kién thức toán, khoa hoc, kỹ thuật, công nghệ, chinh tri, luật 5 pháp, xã hội vào giải quyết cdc vấn dé phức tap của ngành cong nghệ thông tin. Sinh viên tốt nghiệp có thể: .1 Mô hình hóa các tình huéng, lựa chon đú

## 5️⃣ LLM Configuration - Gemini API

In [9]:
class LLMManager:
    """Manage LLM initialization and configuration"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self.llm = None
    
    def initialize_gemini(self, api_key: Optional[str] = None) -> ChatGoogleGenerativeAI:
        """Initialize Google Gemini LLM"""
        api_key = api_key or self.config.GOOGLE_API_KEY
        
        if not api_key or api_key == "YOUR_API_KEY_HERE":
            raise ValueError(
                "❌ Google API Key not configured!\n"
                "Get your free API key at: https://makersuite.google.com/app/apikey\n"
                "Then set environment variable GOOGLE_API_KEY or pass api_key directly"
            )
        
        print(f"\n⏳ Initializing Gemini ({self.config.LLM_MODEL})...")
        
        self.llm = ChatGoogleGenerativeAI(
            model=self.config.LLM_MODEL,
            google_api_key=api_key,
            temperature=self.config.LLM_TEMPERATURE,
            max_output_tokens=self.config.LLM_MAX_TOKENS,
            convert_system_message_to_human=True
        )
        
        print("✅ Gemini LLM initialized successfully")
        print(f"   Model: {self.config.LLM_MODEL}")
        print(f"   Temperature: {self.config.LLM_TEMPERATURE}")
        print(f"   Max tokens: {self.config.LLM_MAX_TOKENS}")
        
        return self.llm

# Initialize LLM manager
llm_manager = LLMManager(config)

# Initialize Gemini with API key from environment variable
# If GOOGLE_API_KEY is not set in .env, you can pass it directly:
# llm = llm_manager.initialize_gemini("your-api-key-here")

try:
    llm = llm_manager.initialize_gemini()  # Auto-load from config.GOOGLE_API_KEY
except ValueError as e:
    print(str(e))
    print("\n💡 Solutions:")
    print("   1. Create .env file with: GOOGLE_API_KEY=your-key")
    print("   2. Or pass directly: llm = llm_manager.initialize_gemini('your-key')")


⏳ Initializing Gemini (gemini-2.5-flash)...
✅ Gemini LLM initialized successfully
   Model: gemini-2.5-flash
   Temperature: 0.2
   Max tokens: 1024


## 6️⃣ RAG Pipeline - Chain Creation

In [10]:
class RAGPipeline:
    """Complete RAG pipeline with retrieval and generation"""
    
    NOT_FOUND_FALLBACK = "Tôi không tìm thấy thông tin này trong tài liệu"
    
    def __init__(self, llm, vector_store, config: RAGConfig):
        self.llm = llm
        self.vector_store = vector_store
        self.config = config
        self.qa_chain = None
        self._build_prompt_template()
        self._create_qa_chain()
    
    @staticmethod
    def _normalize_text(text: str) -> str:
        lowered = text.casefold()
        without_marks = "".join(
            ch for ch in unicodedata.normalize("NFD", lowered) if unicodedata.category(ch) != "Mn"
        )
        return re.sub(r"[^a-z0-9]+", " ", without_marks).strip()
    
    def _is_not_found_answer(self, answer: str) -> bool:
        return self._normalize_text(answer) == self._normalize_text(self.NOT_FOUND_FALLBACK)
    
    def _build_retriever(self, k: Optional[int] = None):
        effective_k = int(k or self.config.RETRIEVAL_K)
        fetch_k = max(effective_k * 4, effective_k + 8)
        # MMR helps reduce duplicate/noisy chunks from OCR-heavy corpora
        return self.vector_store.as_retriever(
            search_type="mmr",
            search_kwargs={"k": effective_k, "fetch_k": fetch_k, "lambda_mult": 0.7},
        )
    
    def _build_prompt_template(self):
        """Create optimized prompt template"""
        template = """Bạn là trợ lý AI chuyên nghiệp, chuyên gia về tài liệu đào tạo.

NHIỆM VỤ: Trả lời câu hỏi của người dùng dựa trên thông tin từ tài liệu được cung cấp.

THÔNG TIN TỪ TÀI LIỆU:
{context}

CÂU HỎI: {question}

YÊU CẦU:
1. Trả lời chính xác, dựa hoàn toàn vào thông tin được cung cấp
2. Trả lời ngắn gọn, rõ ràng bằng tiếng Việt
3. Nếu không tìm thấy thông tin, hãy trả lời: "Tôi không tìm thấy thông tin này trong tài liệu"
4. Không bịa đặt thông tin không có trong tài liệu
5. Sử dụng bullet points nếu cần liệt kê
6. Nếu câu hỏi không liên quan đến tài liệu, trả lời: "Câu hỏi này không liên quan đến tài liệu đã cho"

TRẢ LỜI:"""
        
        self.prompt = PromptTemplate(
            template=template,
            input_variables=["context", "question"]
        )
    
    def _create_qa_chain(self, k: Optional[int] = None):
        """Create RetrievalQA chain"""
        effective_k = int(k or self.config.RETRIEVAL_K)
        print(f"\n Building RAG chain (k={effective_k}, search=mmr)...")
        
        self.qa_chain = RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type="stuff",
            retriever=self._build_retriever(effective_k),
            chain_type_kwargs={"prompt": self.prompt},
            return_source_documents=True
        )
        
        print(" RAG chain created successfully")
    
    def query(self, question: str, verbose: bool = True) -> Dict:
        """Execute query through RAG pipeline"""
        if not self.qa_chain:
            raise RuntimeError("QA chain not initialized")
        
        if verbose:
            print(f"\n{'='*70}")
            print(f" QUESTION: {question}")
            print(f"{'='*70}")
            print(" Processing...\n")
        
        try:
            result = self.qa_chain.invoke({"query": question})
            answer_text = str(result.get("result", "")).strip()
            source_docs = result.get("source_documents", [])
            
            # Retry once with broader retrieval if model falls back while context exists.
            if self._is_not_found_answer(answer_text) and source_docs:
                retry_k = max(self.config.RETRIEVAL_K + 3, 8)
                if verbose:
                    print(f"⚠️ Fallback detected with {len(source_docs)} source docs -> retry with k={retry_k}")
                retry_chain = RetrievalQA.from_chain_type(
                    llm=self.llm,
                    chain_type="stuff",
                    retriever=self._build_retriever(retry_k),
                    chain_type_kwargs={"prompt": self.prompt},
                    return_source_documents=True
                )
                retry_result = retry_chain.invoke({"query": question})
                retry_answer = str(retry_result.get("result", "")).strip()
                if retry_answer and not self._is_not_found_answer(retry_answer):
                    result = retry_result
                    answer_text = retry_answer
                    source_docs = result.get("source_documents", [])
                    if verbose:
                        print("✅ Retry recovered a grounded answer")
            
            if not answer_text:
                answer_text = self.NOT_FOUND_FALLBACK
                result["result"] = answer_text
            
            if verbose:
                print(f"{'='*70}")
                print(" ANSWER:")
                print(f"{'='*70}")
                print(result['result'])
                
                print(f"\n{'='*70}")
                print(" SOURCE DOCUMENTS:")
                print(f"{'='*70}")
                for i, doc in enumerate(source_docs, 1):
                    print(f"\n[Document {i}]")
                    print(doc.page_content[:200] + "...")
                print(f"\n{'='*70}\n")
            
            return result
            
        except Exception as e:
            print(f" Error during query: {e}")
            raise
    
    def batch_query(self, questions: List[str]) -> List[Dict]:
        """Process multiple questions"""
        results = []
        for i, q in enumerate(questions, 1):
            print(f"\n🔄 Processing question {i}/{len(questions)}")
            result = self.query(q, verbose=False)
            results.append(result)
            print(f"✅ Question {i} completed")
        return results

# Initialize RAG pipeline
if 'llm' in dir() and llm is not None:
    rag_pipeline = RAGPipeline(llm, vector_store, config)
    print("\n🎉 RAG System is ready!")
else:
    print("⚠️ Please initialize LLM first (set API key in previous cell)")


 Building RAG chain (k=6, search=mmr)...
 RAG chain created successfully

🎉 RAG System is ready!


## 7️⃣ Usage Examples & Testing

In [11]:
# Single query example
question = "Mục tiêu đào tạo của ngành Công nghệ thông tin là gì?"
result = rag_pipeline.query(question)


 QUESTION: Mục tiêu đào tạo của ngành Công nghệ thông tin là gì?
 Processing...

 ANSWER:
Mục tiêu đào tạo của ngành Công nghệ thông tin bao gồm:

*   **Mục tiêu đào tạo chung:**
    *   Đào tạo người học có phẩm chất chính trị, đạo đức nghề nghiệp, trách nhiệm với công việc, kiến thức cơ sở vững vàng, chuyên môn tốt.
    *   Đào tạo nhân lực chất lượng cao có khả năng ứng dụng công nghệ thông tin để giải quyết các bài toán trong thực tế.
*   **Mục tiêu đào tạo cụ thể:**
    *   PO1. Thực hiện công việc của một kỹ sư CNTT một cách chuyên nghiệp và đảm bảo các chuẩn mực đạo đức nghề nghiệp.
    *   PO2. Nghiên cứu đổi mới và chuyển giao công nghệ, luôn đổi mới và đóng góp vào sự phát triển của CNTT.

 SOURCE DOCUMENTS:

[Document 1]
. Cơ hội học tập, nâng cao trình độ sau tốt nghiệp Có khả năng tự nghiên cứu và cập nhật công nghệ mới về lĩnh vực công nghệ thông tin dé nang cao trình độ va đáp tng yêu cau công việc thực tiễn. Có đ...

[Document 2]
. 4.2. Sir mang Dao tao nguén nhân luc 

In [12]:
# Batch queries example
questions = [
    "Giới thiệu về chương trình đào tạo ngành Công nghệ thông tin?",
    "Thời gian đào tạo là bao lâu?",
    "Sinh viên sẽ học những môn học nào?",
    "Cơ hội nghề nghiệp sau khi tốt nghiệp?"
]

print("\n Processing batch queries...\n")
results = rag_pipeline.batch_query(questions)

print("\n" + "="*70)
print(" BATCH RESULTS SUMMARY")
print("="*70)
for i, (q, r) in enumerate(zip(questions, results), 1):
    print(f"\nQ{i}: {q}")
    print(f"A{i}: {r['result'][:150]}...")
    print("-" * 70)


 Processing batch queries...


🔄 Processing question 1/4
✅ Question 1 completed

🔄 Processing question 2/4
✅ Question 2 completed

🔄 Processing question 3/4
✅ Question 3 completed

🔄 Processing question 4/4


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 58.386711299s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_value: 5
}
, retry_

✅ Question 4 completed

 BATCH RESULTS SUMMARY

Q1: Giới thiệu về chương trình đào tạo ngành Công nghệ thông tin?
A1: Chương trình đào tạo ngành Công nghệ thông tin là chương trình kỹ sư CNTT.

Chương trình này cụ thể hóa các kiến thức thành các chuẩn đầu ra cấp CTĐT ...
----------------------------------------------------------------------

Q2: Thời gian đào tạo là bao lâu?
A2: Thời gian đào tạo được quy định như sau:

*   **Chương trình đào tạo đại học theo hình thức VLVH:**
    *   Bậc cử nhân: tương đương 4...
----------------------------------------------------------------------

Q3: Sinh viên sẽ học những môn học nào?
A3: Sinh viên sẽ học các môn học sau:
*   Cơ sở lập trình
*   Chủ nghĩa xã hội khoa học
*   Tư tưởng Hồ Chí Minh
*   Công nghệ tri thức
*   Học sâu...
----------------------------------------------------------------------

Q4: Cơ hội nghề nghiệp sau khi tốt nghiệp?
A4: Tôi không tìm thấy thông tin này trong tài liệu....
----------------------------------------------

In [18]:
# Interactive query function
def ask(question: str):
    """Convenience function for quick queries"""
    return rag_pipeline.query(question)

# Example usage:
# ask("Chương trình học có những học phần nào?")

## 8️⃣ System Utilities & Maintenance

In [19]:
# System information
def print_system_info():
    """Display system configuration and status"""
    print("\n" + "="*70)
    print(" RAG SYSTEM INFORMATION")
    print("="*70)
    print(f"\n Configuration:")
    print(f"   Base Directory: {config.BASE_DIR}")
    print(f"   PDF Directory: {config.PDF_DIR}")
    print(f"   Vector Store: {config.VECTOR_STORE_DIR}")
    print(f"\n Models:")
    print(f"   Embedding: {config.EMBEDDING_MODEL}")
    print(f"   LLM: {config.LLM_MODEL}")
    print(f"\n Parameters:")
    print(f"   Chunk Size: {config.CHUNK_SIZE}")
    print(f"   Chunk Overlap: {config.CHUNK_OVERLAP}")
    print(f"   Retrieval K: {config.RETRIEVAL_K}")
    print(f"\n Data:")
    if 'chunks' in dir():
        print(f"   Total Chunks: {len(chunks)}")
        print(f"   Total Characters: {sum(len(c) for c in chunks):,}")
    print(f"\n Status: {'Active' if 'rag_pipeline' in dir() else 'Not initialized'}")
    print("="*70)

print_system_info()


 RAG SYSTEM INFORMATION

 Configuration:
   Base Directory: a:\RAG-SGU
   PDF Directory: a:\RAG-SGU\File_PDFs
   Vector Store: a:\RAG-SGU\vector_store

 Models:
   Embedding: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
   LLM: gemini-2.5-flash

 Parameters:
   Chunk Size: 1000
   Chunk Overlap: 200
   Retrieval K: 3

 Data:

 Status: Not initialized


In [20]:
# Reload vector store from disk (if previously saved)
def reload_system():
    """Reload vector store without reprocessing PDFs"""
    global vector_store, rag_pipeline
    
    try:
        vector_store = vector_manager.load_vector_store()
        
        if 'llm' in dir() and llm is not None:
            rag_pipeline = RAGPipeline(llm, vector_store, config)
            print(" System reloaded successfully")
        else:
            print(" Vector store loaded, but LLM not initialized")
    except Exception as e:
        print(f"❌ Reload failed: {e}")

# Uncomment to reload:
# reload_system()